In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier
)
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error
)


from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

from sklearn.preprocessing import StandardScaler
import optuna


In [ ]:
# بفهم واشوف الداتا
Churn = pd.read_csv("Churn_Modelling.csv")
Churn.head()
Churn.info()
Churn.shape
Churn.isnull().sum() #no null values
Churn.duplicated().sum

In [ ]:
# بشوف العميل Exited -> 1 يبقي ساب البنك 
Churn["Exited"].value_counts() # 2037 person Exited

y=Churn['Exited']    
x=Churn.drop("Exited",axis=1)
x = x.drop(["RowNumber", "CustomerId", "Surname"], axis=1) #ملهومش لازمه عندي

In [ ]:
x.select_dtypes(include="object").columns #categorical columns
x = pd.get_dummies(x, drop_first=True) #حولته ل true , false يعني 1,0
x.head()

In [ ]:
X_train,X_test,Y_train,Y_test=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y
)
model=RandomForestClassifier(random_state=42)
model.fit(X_train,Y_train)
Y_predict=model.predict(X_test)


In [ ]:
def objective(trial):

    model = RandomForestClassifier(
        n_estimators=trial.suggest_int("n_estimators", 100, 500),
        max_depth=trial.suggest_int("max_depth", 3, 20),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 10),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 5),
        max_features=trial.suggest_categorical(
            "max_features",
            ["sqrt", "log2", None]
        ),
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, Y_train)

    pred = model.predict(X_test)

    return accuracy_score(Y_test, pred)


study = optuna.create_study(direction="maximize")

study.optimize(
    objective,
    n_trials=100
)

print("Best Accuracy:", study.best_value)
print("Best Parameters:", study.best_params)

In [ ]:
# Best Optuna Model

optuna_model = RandomForestClassifier(
    **study.best_params,
    random_state=42,
    n_jobs=-1
)

optuna_model.fit(X_train, Y_train)

optuna_predict = optuna_model.predict(X_test)

optuna_accuracy = accuracy_score(
    Y_test,
    optuna_predict
)

print("Optuna Churn Model")
print("Test Accuracy:", optuna_accuracy)

print(
    classification_report(
        Y_test,
        optuna_predict
    )
)


In [ ]:
space = {
    "n_estimators": hp.quniform(
        "n_estimators", 100, 500, 10
    ),

    "max_depth": hp.quniform(
        "max_depth", 3, 20, 1
    ),

    "min_samples_split": hp.quniform(
        "min_samples_split", 2, 10, 1
    ),

    "min_samples_leaf": hp.quniform(
        "min_samples_leaf", 1, 5, 1
    ),

    "max_features": hp.choice(
        "max_features",
        ["sqrt", "log2", None]
    )
}

In [ ]:
def hyperopt_objective(params):

    params["n_estimators"] = int(params["n_estimators"])
    params["max_depth"] = int(params["max_depth"])
    params["min_samples_split"] = int(params["min_samples_split"])
    params["min_samples_leaf"] = int(params["min_samples_leaf"])

    model = RandomForestClassifier(
        **params,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, Y_train)

    pred = model.predict(X_test)

    accuracy = accuracy_score(Y_test, pred)

    return {
        "loss": -accuracy,
        "status": STATUS_OK
    }

In [ ]:
trials = Trials()

best_hyperopt = fmin(
    fn=hyperopt_objective,
    space=space,
    algo=tpe.suggest,
    max_evals=100,
    trials=trials
)

In [ ]:

if best_hyperopt["max_features"] == 0:
    max_features = "sqrt"
else:
    max_features = "log2"


hyperopt_model = RandomForestClassifier(
    n_estimators=int(best_hyperopt["n_estimators"]),
    max_depth=int(best_hyperopt["max_depth"]),
    min_samples_split=int(best_hyperopt["min_samples_split"]),
    min_samples_leaf=int(best_hyperopt["min_samples_leaf"]),
    max_features=max_features,
    random_state=42,
    n_jobs=-1
)

hyperopt_model.fit(X_train, Y_train)

hyperopt_predict = hyperopt_model.predict(X_test)

hyperopt_accuracy = accuracy_score(
    Y_test,
    hyperopt_predict
)

print("Hyperopt Churn Model")
print("Test Accuracy:", hyperopt_accuracy)

print(
    classification_report(
        Y_test,
        hyperopt_predict
    )
)